In [41]:
# input
path2pic = 'reference_images'
path2triplet = 'triplet_dataset'
triplet_dataset = 'triplets_large_final_correctednc_correctedorder.csv'
triplet_index = 'unique_id.txt'
img_extension = '.jpg'

# output
random_index = 'random_index.txt'
triplet_sample_index = 'triplet_sample_index.csv'
# index
triplet_train_index = 'triplet_train_index.csv'
triplet_valid_index = 'triplet_valid_index.csv'
triplet_test_index = 'triplet_test_index.csv'
# CLIP
clip_encoder_index = 'clip_index.npy'
clip_train_csv = 'clip_train.csv'
clip_valid_csv = 'clip_valid.csv'
clip_test_csv = 'clip_test.csv'
# Pixel+PCA
pixel_pca_index = 'pixel_index.npy'
pixel_train_csv = 'pixel_train.csv'
pixel_valid_csv = 'pixel_valid.csv'
pixel_test_csv = 'pixel_test.csv'

# EDA procedure

## (Data preprocessing) Load and organize data

In [2]:
import os
root_dir = os.path.dirname(os.getcwd())

pic_dir = os.path.join(root_dir, path2pic)
triplet_data = os.path.join(root_dir, path2triplet, triplet_dataset)
triplet_index = os.path.join(root_dir, path2triplet, triplet_index)
random_index = os.path.join(root_dir, path2triplet, random_index)

### Get image name list

In [3]:
with open(triplet_index) as f:
    image_names = [line.strip()+img_extension for line in f.readlines()]
image_names[0:3]

['aardvark.jpg', 'abacus.jpg', 'accordion.jpg']

In [4]:
len(image_names)

1854

In [5]:
picture_path = [os.path.join(pic_dir, image_name) for image_name in image_names]
picture_path[0:5]

['/project/wilma/katyzhang/MACS30100/reference_images/aardvark.jpg',
 '/project/wilma/katyzhang/MACS30100/reference_images/abacus.jpg',
 '/project/wilma/katyzhang/MACS30100/reference_images/accordion.jpg',
 '/project/wilma/katyzhang/MACS30100/reference_images/acorn.jpg',
 '/project/wilma/katyzhang/MACS30100/reference_images/air_conditioner.jpg']

### Get triplet data

#### Original dataset
We would use 4 columns from this dataset:
- image1
- image2
- image3
- choice

In [6]:
import pandas as pd

# read data
triplet = pd.read_csv(triplet_data, sep="\t")
triplet.head(5)

,image1,image2,image3,choice,RT,noise_ceiling,subject_id,HIT_nr,trial_nr,age,gender,date,time,dataset
0,1245,1050,494,3,14457,0,BMWLG5ZY79VLA,1,1,NaN,other,2018-02-05,09:31:16,1
1,888,1788,1250,3,5043,0,BMWLG5ZY79VLA,1,2,NaN,other,2018-02-05,09:31:16,1
2,1256,734,946,2,6605,0,BMWLG5ZY79VLA,1,3,NaN,other,2018-02-05,09:31:16,1
3,1069,1037,953,1,6177,0,BMWLG5ZY79VLA,1,4,NaN,other,2018-02-05,09:31:16,1
4,1693,803,963,3,2327,0,BMWLG5ZY79VLA,1,5,NaN,other,2018-02-05,09:31:16,1


##### Original plan: Permutation of a, b, and c
In this project, each sample is defined as a triplet of a, b, and c, representing the items in odd-one-out judgment. They function as positional indices used to distinguish the three slots within a triplet, instead of a new set of stimuli. 
Changing the ordering of samples aims to avoid the model’s learning of positional biases, or treating the position itself as informative. To prevent this, we permute the order of a, b, and c during the data processing and exploratory data analysis. The permutations of the same triplet represent an equivalent set (e.g., (a, b, c), (b, a, c), (c, b, a)) and should yield the same interpretations.

In [ ]:
# from itertools import permutations

# def build_perm_dict():
#     perm_dict = {} # Initailize the dictionary
#     # 6 list, each value represent the original element's position
#     perms = list(permutations([1, 2, 3]))
#     # print(perms)

#     # Loop: each choice
#     for old_choice in [1, 2, 3]:
#         out = []
#         # Loop: through the permutation
#         for p in perms:
#             # value in p represents the original element's position
#             new_choice = p.index(old_choice) + 1 # to avoid the problem of 0 when training the model
#             out.append((p[0], p[1], p[2], new_choice)) # append the data
#         perm_dict[str(old_choice)] = out
#     return perm_dict

# perm_dict = build_perm_dict()
# perm_dict["2"]

[(1, 2, 3, 2),
 (1, 3, 2, 3),
 (2, 1, 3, 1),
 (2, 3, 1, 1),
 (3, 1, 2, 3),
 (3, 2, 1, 2)]

In [ ]:
# import pandas as pd

# def apply_perm_expand(df, perm_dict):
#     rows = []
#     # Loop: for each row in dataframe
#     for _, r in df.iterrows():
#         # only care about 4 columns from the original dataframe
#         r = r[["image1", "image2", "image3", "choice"]]
#         c = str(int(r["choice"]))  # get choice: "1"/"2"/"3"
#         # Loop: through perm_dict
#         for (A_, B_, C_, new_c) in perm_dict[c]:
#             # reorganize
#             old_imgs = [r["image1"], r["image2"], r["image3"]]
#             new_imgs = [old_imgs[A_-1], old_imgs[B_-1], old_imgs[C_-1]]

#             # store in the right order
#             new_r = r.copy()
#             new_r["image1"], new_r["image2"], new_r["image3"] = new_imgs
#             new_r["choice"] = new_c

#             # record the permutation
#             new_r["perm"] = (A_, B_, C_)
#             rows.append(new_r)

#     return pd.DataFrame(rows).reset_index(drop=True)

# triplet_perm = apply_perm_expand(df_train, perm_dict)

In [ ]:
# triplet_perm.to_csv(triplet_train_index, index=False)

In [ ]:
# triplet_perm = pd.read_csv(triplet_train_index)
# print(len(triplet_perm))
# triplet_perm.head(10)

1260000


,image1,image2,image3,choice,perm
0,675,416,1522,1,"(1, 2, 3)"
1,675,1522,416,1,"(1, 3, 2)"
2,416,675,1522,2,"(2, 1, 3)"
3,416,1522,675,3,"(2, 3, 1)"
4,1522,675,416,2,"(3, 1, 2)"
5,1522,416,675,3,"(3, 2, 1)"
6,947,1314,337,2,"(1, 2, 3)"
7,947,337,1314,3,"(1, 3, 2)"
8,1314,947,337,1,"(2, 1, 3)"
9,1314,337,947,1,"(2, 3, 1)"


#### (Error removal) Remove the duplicate from original dataset
There are several within-subject duplicate to check participants' engagement levels, we would directly remove those duplicates

In [7]:
import numpy as np

# remove the duplicate
cols = ["image1", "image2", "image3"]

# create an order-invariant key by sorting the three IDs row-wise
key = pd.DataFrame(np.sort(triplet[cols].to_numpy(), axis=1), columns=[f"k{i}" for i in range(3)])

triplet_unique_any_order = triplet.loc[~key.duplicated()].reset_index(drop=True)
triplet_unique_any_order.head(5)

,image1,image2,image3,choice,RT,noise_ceiling,subject_id,HIT_nr,trial_nr,age,gender,date,time,dataset
0,1245,1050,494,3,14457,0,BMWLG5ZY79VLA,1,1,NaN,other,2018-02-05,09:31:16,1
1,888,1788,1250,3,5043,0,BMWLG5ZY79VLA,1,2,NaN,other,2018-02-05,09:31:16,1
2,1256,734,946,2,6605,0,BMWLG5ZY79VLA,1,3,NaN,other,2018-02-05,09:31:16,1
3,1069,1037,953,1,6177,0,BMWLG5ZY79VLA,1,4,NaN,other,2018-02-05,09:31:16,1
4,1693,803,963,3,2327,0,BMWLG5ZY79VLA,1,5,NaN,other,2018-02-05,09:31:16,1


In [8]:
len(triplet_unique_any_order)

4567526

#### (Distribution Check) Extract random sample from whole dataset
Randomly select 50,000 data, cause the dataset is too large: 469,9160

In [9]:
sample_n = 50000

In [10]:
# set tag to detect: (1) all images included? (2) contain duplicates? (3) balanced distribution?
all_images_included_tag = 0
no_duplicate_tag = 0
distribution_tag = 0

In [ ]:
# # Randomly select 50,000 data, cause the dataset is too large: 469,9160
# i = 0
# while not (all_images_included_tag and no_duplicate_tag and distribution_tag):
#     # Tag how many times of trying
#     i += 1
#     print("--"*5, f"try {i}: ", "--"*5)

#     random_index_label = triplet_unique_any_order.sample(n=sample_n).index
#     # to make sure that all the images are included
#     image_set = set() 
#     # detect duplicate
#     duplicate_detect_set = set()
#     for index in random_index_label:
#         img1 = triplet_unique_any_order["image1"].iloc[index]
#         img2 = triplet_unique_any_order["image2"].iloc[index]
#         img3 = triplet_unique_any_order["image3"].iloc[index]
#         image_set.add(img1)
#         image_set.add(img2)
#         image_set.add(img3)
#         duplicate_detect_set.add((img1, img2, img3))

#     print(random_index_label.unique())

#     all_images_included_tag = (len(image_set)==len(image_names))
#     print("All images (1854) are included in the sample:", all_images_included_tag)

#     no_duplicate_tag = (len(duplicate_detect_set)==len(random_index_label))
#     print("No duplicate images combinations in the sample:", no_duplicate_tag)

#     # if index is valid, then check the distribution
#     if all_images_included_tag and no_duplicate_tag:
#         triplet_sample = triplet_unique_any_order[["image1", "image2", "image3", "choice"]].iloc[random_index_label]
#         distribution_tag = (triplet_sample['choice'].value_counts()/sample_n> 0.32999).all()
#         print("Balanced distribution:", distribution_tag)

---------- try 1:  ----------
Index([2286584, 1816818, 3246105, 4094636, 4484500, 1333339, 4559183, 2527085,
        663272, 2237497,
       ...
       2811376, 4060886,  735013, 4538406, 2763277, 2235415, 3166665, 3701074,
       1662620, 1643074],
      dtype='int64', length=50000)
All images (1854) are included in the sample: True
No duplicate images combinations in the sample: True
Balanced distribution: False
---------- try 2:  ----------
Index([2771743,  807546, 4291790, 3809548, 2336926,  890943, 2448228,  522981,
       1326233, 2882549,
       ...
        755531,   20405, 3667735, 1420652, 2200994, 2981171, 4290830, 2981578,
       3792089, 2885047],
      dtype='int64', length=50000)
All images (1854) are included in the sample: True
No duplicate images combinations in the sample: True
Balanced distribution: False
---------- try 3:  ----------
Index([4208858, 3804838,  305699, 3931447, 1798471,  540489, 2503892, 1851074,
       3981925, 2514954,
       ...
       3967748, 418

In [ ]:
# with open(random_index, mode = "w") as f:
#     for index in random_index_label:
#         f.write(str(index) + '\n')

In [13]:
with open(random_index) as f:
    index_selected = [line.strip() for line in f.readlines()]
    
len(index_selected)

50000

In [ ]:
# triplet_sample = triplet_unique_any_order[["image1", "image2", "image3", "choice"]].iloc[index_selected]
# triplet_sample.to_csv(triplet_sample_index, index=False)

In [20]:
triplet_sample = pd.read_csv(triplet_sample_index)

In [21]:
triplet_sample['choice'].value_counts()

choice
2    16900
1    16561
3    16539
Name: count, dtype: int64

In [22]:
triplet_sample.head(5)

,image1,image2,image3,choice
0,731,1152,1225,3
1,1498,1790,152,3
2,899,1737,834,1
3,1167,793,1309,1
4,720,1843,753,3


## (Split Dataset) Split data into 3 sets
- 70% for training set
- 15% for validation set
- 15% for testing set

In [23]:
ratio_train = 0.7
ratio_test = 0.15

ratio_valid = 1 - ratio_train - ratio_test
ratio_ttsplit = ratio_test
ratio_tvsplit = ratio_valid / (ratio_train + ratio_valid)

In [24]:
from sklearn.model_selection import train_test_split

triplet_sample_X = triplet_sample[["image1", "image2", "image3"]]
triplet_sample_y = triplet_sample["choice"]
X_main, X_test, y_main, y_test = train_test_split(triplet_sample_X, triplet_sample_y, test_size=ratio_ttsplit, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_main, y_main, test_size=ratio_tvsplit, random_state=42)
df_train = X_train.join(y_train)
df_valid = X_val.join(y_val)
df_test = X_test.join(y_test)

In [ ]:
# df_train.to_csv(triplet_train_index, index=False)
# df_valid.to_csv(triplet_valid_index, index=False)
# df_test.to_csv(triplet_test_index, index=False)

## (Data transformation) Feature generation

### Convert pictures using CLIP encoder
Using [CLIP](https://openai.com/index/clip/) to convert one image into a vector (512, 1) to represent this image's positions in CLIP "semantic space".
- In other word, embedding the pictures using the semantic information

In [87]:
import clip
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load(
    "ViT-B/32",
    device=device,
    download_root="/project/wilma/katyzhang/CLIP"
)
model.eval()

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
    (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): Sequential(
        (0): ResidualAttentionBlock(
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=768, out_features=3072, bias=True)
            (gelu): QuickGELU()
            (c_proj): Linear(in_features=3072, out_features=768, bias=True)
          )
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        )
        (1): ResidualAttentionBlock(
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          

In [88]:
import PIL.Image as Image
import json

def encode_image_openclip(image_paths, device=device, dtype=np.float32):
    N = len(image_paths)
    X = np.zeros((N, 512), dtype=dtype)

    for i, p in enumerate(image_paths):
        image = Image.open(p).convert("RGB")
        
        image_tensor = preprocess(image).unsqueeze(0).to(device)
        with torch.no_grad():
            image_features = model.encode_image(image_tensor)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)

        if (i + 1) % 200 == 0:
            print(f"Loaded {i+1}/{N}")

        X[i] = image_features
    
    return X

clip_index = encode_image_openclip(picture_path)
print("")
print("clip_index:", clip_index.shape)

Loaded 200/1854
Loaded 400/1854
Loaded 600/1854
Loaded 800/1854
Loaded 1000/1854
Loaded 1200/1854
Loaded 1400/1854
Loaded 1600/1854
Loaded 1800/1854

clip_index: (1854, 512)


In [ ]:
# np.save(clip_encoder_index, clip_index)

### Convert pictures into pixels
To match the output dimension of CLIP (512, 1), we use PCA (dimention = 512) to capture the picture's pixel level charateristics. To avoid memory issue, we would resize the pictures into (64, 64, 3)
- In other word, embedding the pictures using the perceptual information

In [89]:
import json
import numpy as np
from PIL import Image
from sklearn.decomposition import PCA

def load_pixels_matrix(image_paths, size=(64, 64), dtype=np.float32):
    """
    resize the pictures into (64, 64,3)
    """
    N = len(image_paths)
    D = size[0] * size[1] * 3
    X = np.zeros((N, D), dtype=dtype)

    for i, p in enumerate(image_paths):
        img = Image.open(p).convert("RGB").resize(size, Image.BICUBIC)
        arr = np.asarray(img, dtype=np.uint8)               # (64,64,3)
        X[i] = (arr.reshape(-1).astype(dtype) / 255.0)      # (12288,)

        if (i + 1) % 200 == 0:
            print(f"Loaded {i+1}/{N}")

    return X

# ---- usage ----
# image_paths = [...]  # list of 1854 file paths

X = load_pixels_matrix(picture_path, size=(64, 64))

pca = PCA(n_components=512, random_state=0)
pixel_index = pca.fit_transform(X)   # (N,512)

print("")
print("X:", X.shape, "pixel_index:", pixel_index.shape)

Loaded 200/1854
Loaded 400/1854
Loaded 600/1854
Loaded 800/1854
Loaded 1000/1854
Loaded 1200/1854
Loaded 1400/1854
Loaded 1600/1854
Loaded 1800/1854

X: (1854, 12288) pixel_index: (1854, 512)


In [ ]:
# np.save(pixel_pca_index, pixel_index)

## Build up the dataset

- For each feature(dimension), the absolute distance will be calculated among 3 pictures, to get the most dissimilar picture.
- That picture's order would be seen as the input for that feature.

In [42]:
# read df_train, df_valid, df_test
import pandas as pd

df_train = pd.read_csv(triplet_train_index)
df_valid = pd.read_csv(triplet_valid_index)
df_test = pd.read_csv(triplet_test_index)

### (Semantic dataset) CLIP
- feature(512): Dimention
- row: each "odd-one-out" task

In [43]:
import pandas as pd
import numpy as np

def farthest_idx(X):
    """
    X: array-like, shape (N,3)
    returns:
      idx_1based: shape (N,) values in {1,2,3}
      farthest_val: shape (N,)
    """
    X = np.asarray(X).T

    d01 = np.abs(X[:, 0] - X[:, 1])
    d02 = np.abs(X[:, 0] - X[:, 2])
    d12 = np.abs(X[:, 1] - X[:, 2])

    scores = np.stack([
        d01 + d02,  # element 1 vs others
        d01 + d12,  # element 2 vs others
        d02 + d12   # element 3 vs others
    ], axis=1)  # (N,3)

    idx0 = np.argmax(scores, axis=1)              # (N,) 0-based
    idx_1based = idx0 + 1                         # (N,) in {1,2,3}
    return idx_1based

clip_encoder = np.load(clip_encoder_index)
dimension = len(clip_encoder[0])

In [ ]:
# clip_train
column_names = [f"D_{i}" for i in range(dimension)] + ["Choice"]
clip_train = pd.DataFrame(columns=column_names)
train_len = len(df_train)

ii = 0
for index, row in df_train.iterrows():
    # get clip vectors for each image
    clip_img1 = clip_encoder[row['image1']-1]
    clip_img2 = clip_encoder[row['image2']-1]
    clip_img3 = clip_encoder[row['image3']-1]
    # calculate the most dissimilar piture for each dimention
    idx = farthest_idx([clip_img1, clip_img2, clip_img3])
    organized_row = np.append(idx, row['choice'])
    # update the dataframe (clip_train)
    clip_train.loc[ii] = organized_row
    ii += 1
    if not ii % 5000:
        print(f"having finished: {ii}/{train_len}")

clip_train.to_csv(clip_train_csv, index=False)
clip_train

having finished: 5000/34999
having finished: 10000/34999
having finished: 15000/34999
having finished: 20000/34999
having finished: 25000/34999
having finished: 30000/34999


,D_0,D_1,D_2,D_3,D_4,D_5,D_6,D_7,D_8,D_9,...,D_503,D_504,D_505,D_506,D_507,D_508,D_509,D_510,D_511,Choice
0,2,2,3,1,3,3,3,3,3,2,...,1,2,1,3,1,2,1,3,1,3
1,2,1,3,2,3,2,3,3,2,1,...,1,2,1,1,1,3,1,2,3,3
2,2,3,1,3,3,2,1,1,3,2,...,2,2,1,2,2,3,1,3,2,3
3,3,3,2,2,2,3,1,3,3,3,...,2,3,2,2,3,3,2,3,1,3
4,2,3,3,1,1,2,1,3,3,2,...,2,1,2,1,1,1,2,3,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34994,3,1,2,3,3,2,3,2,2,1,...,3,3,1,1,3,2,1,1,1,1
34995,2,2,3,1,1,1,2,2,1,3,...,3,2,3,2,2,2,3,2,1,1
34996,2,1,2,2,2,2,2,1,3,1,...,1,2,1,2,1,1,1,3,1,1
34997,3,2,2,2,2,1,2,3,1,3,...,2,3,2,2,2,2,1,2,1,1


In [ ]:
# clip_valid
column_names = [f"D_{i}" for i in range(dimension)] + ["Choice"]
clip_valid = pd.DataFrame(columns=column_names)
valid_len = len(df_valid)

ii = 0
for index, row in df_valid.iterrows():
    # get clip vectors for each image
    clip_img1 = clip_encoder[row['image1']-1]
    clip_img2 = clip_encoder[row['image2']-1]
    clip_img3 = clip_encoder[row['image3']-1]
    # calculate the most dissimilar piture for each dimention
    idx = farthest_idx([clip_img1, clip_img2, clip_img3])
    organized_row = np.append(idx, row['choice'])
    # update the dataframe (clip_valid)
    clip_valid.loc[ii] = organized_row
    ii += 1
    if not ii % 5000:
        print(f"having finished: {ii}/{valid_len}")

clip_valid.to_csv(clip_valid_csv, index=False)
clip_valid

having finished: 5000/7501


,D_0,D_1,D_2,D_3,D_4,D_5,D_6,D_7,D_8,D_9,...,D_503,D_504,D_505,D_506,D_507,D_508,D_509,D_510,D_511,Choice
0,2,2,1,3,3,2,2,3,1,3,...,3,1,1,3,2,3,3,3,3,3
1,3,3,1,1,1,2,2,1,3,1,...,1,3,1,3,3,3,3,2,1,3
2,2,2,2,2,1,1,2,2,3,3,...,3,2,2,3,1,3,3,3,3,3
3,3,1,1,1,2,1,2,2,1,3,...,3,3,1,3,3,2,3,3,3,1
4,3,1,1,2,3,2,1,1,3,1,...,2,2,2,3,3,2,1,3,3,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7496,3,3,1,1,1,1,1,3,3,1,...,3,3,3,3,2,2,3,2,1,3
7497,1,2,3,3,1,1,2,1,2,3,...,2,1,1,3,3,2,2,3,1,3
7498,3,1,1,3,1,3,3,2,2,1,...,3,2,1,3,3,2,1,2,1,1
7499,3,2,1,1,2,1,1,1,2,3,...,3,3,1,1,2,1,2,3,2,2


In [ ]:
# clip_test
column_names = [f"D_{i}" for i in range(dimension)] + ["Choice"]
clip_test = pd.DataFrame(columns=column_names)
test_len = len(df_test)

ii = 0
for index, row in df_test.iterrows():
    # get clip vectors for each image
    clip_img1 = clip_encoder[row['image1']-1]
    clip_img2 = clip_encoder[row['image2']-1]
    clip_img3 = clip_encoder[row['image3']-1]
    # calculate the most dissimilar piture for each dimention
    idx = farthest_idx([clip_img1, clip_img2, clip_img3])
    organized_row = np.append(idx, row['choice'])
    # update the dataframe (clip_test)
    clip_test.loc[ii] = organized_row
    ii += 1
    if not ii % 5000:
        print(f"having finished: {ii}/{test_len}")

clip_test.to_csv(clip_test_csv, index=False)
clip_test

having finished: 5000/7500


,D_0,D_1,D_2,D_3,D_4,D_5,D_6,D_7,D_8,D_9,...,D_503,D_504,D_505,D_506,D_507,D_508,D_509,D_510,D_511,Choice
0,2,2,1,3,2,1,1,2,3,2,...,3,2,2,1,2,3,3,1,3,3
1,3,3,1,2,3,3,3,3,2,1,...,2,3,1,2,3,3,2,3,3,3
2,2,1,2,1,3,3,2,3,3,1,...,1,2,3,1,2,2,3,1,2,1
3,3,3,3,3,2,3,1,3,2,3,...,1,2,3,1,1,3,2,2,2,1
4,3,2,2,1,3,2,3,1,1,1,...,2,3,2,3,1,3,2,1,2,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7495,2,2,2,2,1,2,1,1,1,2,...,1,2,2,2,2,2,2,2,1,1
7496,3,3,2,1,3,3,3,3,3,2,...,1,2,2,2,3,1,1,2,3,3
7497,2,1,1,3,1,3,2,1,2,2,...,1,3,1,2,2,1,1,1,2,2
7498,2,3,2,3,2,2,1,2,2,3,...,2,1,3,1,2,2,2,2,1,3


### (Perceptual dataset) Pixel+PCA

In [47]:
import pandas as pd
import numpy as np

def farthest_idx(X):
    """
    X: array-like, shape (N,3)
    returns:
      idx_1based: shape (N,) values in {1,2,3}
      farthest_val: shape (N,)
    """
    X = np.asarray(X).T

    d01 = np.abs(X[:, 0] - X[:, 1])
    d02 = np.abs(X[:, 0] - X[:, 2])
    d12 = np.abs(X[:, 1] - X[:, 2])

    scores = np.stack([
        d01 + d02,  # element 1 vs others
        d01 + d12,  # element 2 vs others
        d02 + d12   # element 3 vs others
    ], axis=1)  # (N,3)

    idx0 = np.argmax(scores, axis=1)              # (N,) 0-based
    idx_1based = idx0 + 1                         # (N,) in {1,2,3}
    return idx_1based

pixel_encoder = np.load(pixel_pca_index)
dimension = len(pixel_encoder[0])

In [48]:
# pixel_train
column_names = [f"D_{i}" for i in range(dimension)] + ["Choice"]
pixel_train = pd.DataFrame(columns=column_names)
train_len = len(df_train)

ii = 0
for index, row in df_train.iterrows():
    # get pixel vectors for each image
    pixel_img1 = pixel_encoder[row['image1']-1]
    pixel_img2 = pixel_encoder[row['image2']-1]
    pixel_img3 = pixel_encoder[row['image3']-1]
    # calculate the most dissimilar piture for each dimention
    idx = farthest_idx([pixel_img1, pixel_img2, pixel_img3])
    organized_row = np.append(idx, row['choice'])
    # update the dataframe (pixel_train)
    pixel_train.loc[ii] = organized_row
    ii += 1
    if not ii % 5000:
        print(f"having finished: {ii}/{train_len}")

pixel_train.to_csv(pixel_train_csv, index=False)
pixel_train

having finished: 5000/34999
having finished: 10000/34999
having finished: 15000/34999
having finished: 20000/34999
having finished: 25000/34999
having finished: 30000/34999


,D_0,D_1,D_2,D_3,D_4,D_5,D_6,D_7,D_8,D_9,...,D_503,D_504,D_505,D_506,D_507,D_508,D_509,D_510,D_511,Choice
0,3,1,3,2,2,1,3,1,1,3,...,3,3,2,2,2,2,1,3,1,3
1,2,1,3,1,2,1,2,2,2,2,...,1,3,2,2,2,1,3,1,3,3
2,3,2,1,1,3,1,2,2,1,2,...,1,1,2,1,2,1,2,1,1,3
3,1,3,1,1,3,3,3,3,2,3,...,2,3,3,1,1,2,3,1,2,3
4,3,3,3,3,1,2,3,3,1,1,...,2,2,1,3,1,2,3,2,3,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34994,2,1,3,2,3,1,3,2,3,1,...,2,2,2,1,2,2,3,2,2,1
34995,2,1,1,3,3,1,3,1,2,3,...,2,2,2,1,2,3,2,2,1,1
34996,2,2,2,3,2,1,3,2,2,1,...,3,3,1,2,3,2,1,2,1,1
34997,3,3,1,2,3,2,2,3,2,1,...,3,2,3,1,2,2,1,3,1,1


In [49]:
# pixel_valid
column_names = [f"D_{i}" for i in range(dimension)] + ["Choice"]
pixel_valid = pd.DataFrame(columns=column_names)
valid_len = len(df_valid)

ii = 0
for index, row in df_valid.iterrows():
    # get pixel vectors for each image
    pixel_img1 = pixel_encoder[row['image1']-1]
    pixel_img2 = pixel_encoder[row['image2']-1]
    pixel_img3 = pixel_encoder[row['image3']-1]
    # calculate the most dissimilar piture for each dimention
    idx = farthest_idx([pixel_img1, pixel_img2, pixel_img3])
    organized_row = np.append(idx, row['choice'])
    # update the dataframe (pixel_valid)
    pixel_valid.loc[ii] = organized_row
    ii += 1
    if not ii % 5000:
        print(f"having finished: {ii}/{valid_len}")

pixel_valid.to_csv(pixel_valid_csv, index=False)
pixel_valid

having finished: 5000/7501


,D_0,D_1,D_2,D_3,D_4,D_5,D_6,D_7,D_8,D_9,...,D_503,D_504,D_505,D_506,D_507,D_508,D_509,D_510,D_511,Choice
0,1,2,2,2,1,2,2,1,3,1,...,2,3,2,1,1,2,2,1,3,3
1,3,3,2,1,1,3,3,3,3,2,...,3,2,2,3,2,3,3,1,1,3
2,2,1,1,3,2,3,3,1,3,3,...,3,1,1,3,3,1,2,3,1,3
3,3,3,2,1,1,1,1,3,2,2,...,3,3,1,2,1,2,3,1,1,1
4,2,2,2,3,2,2,3,3,3,3,...,2,1,1,2,3,3,3,1,3,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7496,2,3,3,3,1,1,3,1,3,1,...,3,3,3,3,3,1,3,3,3,3
7497,1,3,1,1,1,1,1,1,2,3,...,1,1,1,1,2,2,1,1,2,3
7498,1,1,3,2,2,1,3,2,3,1,...,2,2,3,3,2,1,3,2,2,1
7499,2,1,1,3,3,2,2,3,2,3,...,2,1,1,3,1,2,3,3,3,2


In [50]:
# pixel_test
column_names = [f"D_{i}" for i in range(dimension)] + ["Choice"]
pixel_test = pd.DataFrame(columns=column_names)
test_len = len(df_test)

ii = 0
for index, row in df_test.iterrows():
    # get pixel vectors for each image
    pixel_img1 = pixel_encoder[row['image1']-1]
    pixel_img2 = pixel_encoder[row['image2']-1]
    pixel_img3 = pixel_encoder[row['image3']-1]
    # calculate the most dissimilar piture for each dimention
    idx = farthest_idx([pixel_img1, pixel_img2, pixel_img3])
    organized_row = np.append(idx, row['choice'])
    # update the dataframe (pixel_test)
    pixel_test.loc[ii] = organized_row
    ii += 1
    if not ii % 5000:
        print(f"having finished: {ii}/{test_len}")

pixel_test.to_csv(pixel_test_csv, index=False)
pixel_test

having finished: 5000/7500


,D_0,D_1,D_2,D_3,D_4,D_5,D_6,D_7,D_8,D_9,...,D_503,D_504,D_505,D_506,D_507,D_508,D_509,D_510,D_511,Choice
0,1,2,3,2,2,2,1,1,2,1,...,1,2,2,3,3,1,2,2,3,3
1,1,3,2,2,2,3,2,1,1,2,...,3,1,3,1,1,1,3,3,1,3
2,3,3,1,1,1,2,3,3,1,1,...,1,1,1,3,1,1,3,1,3,1
3,3,1,1,1,1,3,2,2,1,2,...,3,1,3,1,3,1,3,1,2,1
4,2,1,2,2,1,3,3,3,1,2,...,3,3,2,3,3,3,3,3,2,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7495,3,1,3,3,2,1,2,3,2,1,...,2,2,3,2,1,2,1,2,1,1
7496,2,2,1,3,3,2,2,3,2,3,...,1,3,3,1,3,3,2,2,2,3
7497,2,2,3,3,1,3,2,3,2,3,...,3,2,2,2,3,2,2,3,1,2
7498,3,1,1,1,3,3,3,3,2,2,...,2,2,1,1,1,2,3,1,2,3


In [51]:
clip_train.to_csv(clip_train_csv, index=False)
clip_valid.to_csv(clip_valid_csv, index=False)
clip_test.to_csv(clip_test_csv, index=False)
pixel_train.to_csv(pixel_train_csv, index=False)
pixel_valid.to_csv(pixel_valid_csv, index=False)
pixel_test.to_csv(pixel_test_csv, index=False)